#set up

In [ ]:
! pip install py3Dmol
! pip install fairchem-core
!pip install deepchem

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.0/301.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/

In [ ]:
import py3Dmol
import os
import torch
import numpy as np
from fairchem.core import FAIRChemCalculator, pretrained_mlip
import random
import re
import copy
import math
import ase.io
from ase.calculators.calculator import all_changes
from ase.optimize import BFGS
from ase.constraints import FixAtoms
from ase import Atoms, Atom



cpuCount = os.cpu_count()
print(cpuCount)

2


# workflow

In [ ]:
fragments = define_fragments()

In [ ]:
for frag in fragments:
  get_fragment_cutoff(frag, 0.5)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

predictor = pretrained_mlip.get_predict_unit("uma-s-1", device=device)
calculator = FAIRChemCalculator(predictor, task_name="omol")
model = "UMA-OMOL"

In [ ]:
max_values, min_values = get_binding_site_dims(octinoxate_data, 0.0)

In [ ]:
new_molecules, ies = grow_fragments([octinoxate_data], fragments, 100, calculator, max_values, min_values)

In [ ]:
save_xyz_files(new_molecules, fragments, octinoxate_data)

In [ ]:
seperate_poses = get_best_seperate_poses(ies, fragments, octinoxate_data, new_molecules)

In [ ]:
best_poses = get_best_poses(ies, fragments)

In [ ]:
view_frag_pose(0, 0, fragments, octinoxate_data)

In [ ]:
combined_poses, combined_atoms, all_centers, all_names = combine_best_poses(octinoxate_data, fragments, best_poses)

In [ ]:
view_combined_poses(octinoxate_data, fragments, best_poses)

In [ ]:
distances = distance_between_centers(all_centers, all_names)

#

In [ ]:
global octinoxate_data
octinoxate_data = {
    "num_atoms": 44,
    "name": "octinoxate",
    "atoms": ['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'O', 'C', 'H', 'H', 'H', 'C', 'H', 'C', 'H', 'C', 'O', 'O', 'C', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'H'],
    "coords": [
        [-5.945350681818182, 0.23278688636363637, 0.16006995454545458],
        [-5.172542681818181, 1.3434288863636366, 0.4925409545454545],
        [-3.7998686818181815, 1.2793748863636365, 0.39736195454545453],
        [-3.1473386818181814, 0.11847288636363638, -0.02927704545454544],
        [-3.9371006818181815, -0.9809231136363636, -0.35633604545454545],
        [-5.3173046818181815, -0.9356211136363636, -0.26665404545454546],
        [-5.670811681818182, 2.2442728863636363, 0.8226839545454546],
        [-3.2111576818181815, 2.1500078863636363, 0.6591099545454546],
        [-3.4746566818181814, -1.9003121136363634, -0.6895730454545455],
        [-5.890931681818182, -1.8116101136363636, -0.5298930454545454],
        [-7.280922681818182, 0.38014688636363636, 0.2814289545454545],
        [-8.110493681818182, -0.7119621136363636, -0.04155404545454544],
        [-7.905391681818181, -1.5729501136363635, 0.6002269545454546],
        [-9.130393681818182, -0.3767111136363636, 0.12538195454545453],
        [-7.995182681818181, -1.0053401136363636, -1.0883910454545456],
        [-1.6885196818181818, 0.1151798863636364, -0.11031904545454545],
        [-1.2043856818181817, 1.0389918863636365, 0.18673795454545455],
        [-0.9047716818181817, -0.8889731136363637, -0.5051070454545454],
        [-1.2957356818181818, -1.8473391136363635, -0.8184200454545454],
        [0.5661873181818183, -0.8176351136363637, -0.5564920454545454],
        [1.2535253181818184, -1.7482521136363636, -0.9030040454545454],
        [1.0536433181818183, 0.3733468863636364, -0.18911604545454547],
        [2.4877833181818185, 0.5414958863636363, -0.18817904545454545],
        [2.8849863181818183, -0.05701711363636361, -1.0092380454545455],
        [3.0103743181818183, 0.017014886363636388, 1.1451719545454544],
        [2.6290143181818184, -0.9973781136363636, 1.2700839545454545],
        [2.5647793181818184, 0.6213728863636364, 1.9394939545454546],
        [2.7346373181818184, 2.0235678863636366, -0.45721604545454547],
        [1.9231723181818183, 2.3720828863636365, -1.0974950454545456],
        [2.6544743181818182, 2.5825578863636367, 0.47880195454545454],
        [4.060457318181818, 2.3306668863636366, -1.1452480454545455],
        [4.128206318181818, 3.3932688863636367, -1.3785320454545456],
        [4.920967318181819, 2.0744828863636364, -0.5298220454545455],
        [4.147235318181818, 1.7791168863636364, -2.0829780454545452],
        [4.529157318181818, 0.002277886363636388, 1.3127209545454546],
        [4.910828318181818, 1.0251098863636365, 1.3471389545454544],
        [4.745812318181819, -0.4278371136363636, 2.2944199545454547],
        [5.292977318181818, -0.7928611136363636, 0.25283095454545457],
        [6.356103318181819, -0.7603821136363637, 0.5026069545454546],
        [5.199829318181818, -0.30302511363636364, -0.7195700454545454],
        [4.841835318181818, -2.2437721136363633, 0.12912195454545455],
        [5.478644318181819, -2.7908341136363637, -0.5665030454545454],
        [4.892125318181819, -2.7509921136363635, 1.0953669545454545],
        [3.816104318181818, -2.3172971136363634, -0.23438304545454547],
    ],
    "charge": 0,
    "spin": 1,
    "size": 0.0
}

global octocrylene_data
octocrylene_data = {
    "num_atoms": 56,
    "name": "octocrylene",
    "atoms": ['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'C', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'C', 'O', 'C', 'N', 'O', 'C', 'H', 'H', 'C', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'H'],
    "coords": [
        [4.104155589285714, 0.007694517857142813, -1.5863354821428572],
        [5.336717589285714, 0.5401815178571427, -1.9294634821428573],
        [5.963086589285714, 1.444143517857143, -1.085195482142857],
        [5.349236589285714, 1.811554517857143, 0.10228751785714284],
        [4.115592589285715, 1.279299517857143, 0.44331951785714285],
        [3.4810805892857144, 0.3724145178571428, -0.39783548214285713],
        [3.6216165892857144, -0.7053074821428572, -2.243012482142857],
        [5.807968589285714, 0.2482135178571428, -2.858577482142857],
        [6.924160589285714, 1.862514517857143, -1.352839482142857],
        [5.829646589285714, 2.519072517857143, 0.7651115178571428],
        [3.6376195892857144, 1.583221517857143, 1.3660195178571428],
        [2.1541105892857146, -0.2683674821428572, -0.021720482142857156],
        [1.6484565892857144, -0.5481944821428573, -0.9468344821428571],
        [1.2335525892857144, 0.6706295178571429, 0.7331225178571429],
        [0.5853995892857145, 1.6760195178571429, 0.01958551785714284],
        [1.0058025892857145, 0.5796375178571429, 2.100790517857143],
        [-0.2705654107142855, 2.560606517857143, 0.6501995178571428],
        [0.7560275892857145, 1.7601175178571427, -1.0470014821428573],
        [0.14887458928571445, 1.4676995178571428, 2.738772517857143],
        [1.4879995892857145, -0.1910674821428572, 2.6877975178571427],
        [-0.4921184107142854, 2.459415517857143, 2.016877517857143],
        [-0.7720474107142855, 3.3272085178571427, 0.07435651785714284],
        [-0.017559410714285573, 1.3770775178571428, 3.803877517857143],
        [-1.1634814107142857, 3.147541517857143, 2.5126245178571427],
        [2.4236995892857145, -1.5618724821428571, 0.7159165178571428],
        [3.0348865892857146, -1.490914482142857, 1.6112405178571427],
        [1.5260015892857144, -2.7226394821428572, 0.5723825178571429],
        [2.7420085892857147, -2.697602482142857, -0.09256748214285715],
        [1.4281475892857145, -3.687209482142857, 1.673176517857143],
        [1.3152245892857146, -4.405578482142857, 2.5595505178571427],
        [0.4253365892857145, -2.678553482142857, -0.22939848214285719],
        [-0.7487774107142856, -2.1555644821428572, 0.4090415178571428],
        [-1.3652954107142854, -2.9926814821428573, 0.7463065178571429],
        [-0.4555934107142855, -1.579344482142857, 1.2886445178571428],
        [-1.4932554107142852, -1.2526334821428573, -0.5591214821428572],
        [-0.7448914107142856, -0.5688334821428572, -0.9694034821428572],
        [-2.0938674107142856, -2.0265244821428574, -1.7395864821428573],
        [-2.392763410714285, -1.3107414821428571, -2.508177482142857],
        [-1.3036654107142853, -2.6337034821428573, -2.185548482142857],
        [-3.2853164107142856, -2.912874482142857, -1.3948394821428571],
        [-3.039631410714285, -3.6468504821428573, -0.6258794821428572],
        [-4.132982410714286, -2.327521482142857, -1.0356094821428572],
        [-3.6172954107142856, -3.464791482142857, -2.274037482142857],
        [-2.4941164107142852, -0.3970304821428572, 0.21890751785714285],
        [-1.9525834107142854, 0.1111425178571428, 1.022233517857143],
        [-3.2402954107142854, -1.0324114821428572, 0.7063755178571428],
        [-3.1933744107142856, 0.6580625178571429, -0.6267984821428572],
        [-3.8309684107142856, 0.18316451785714277, -1.3781884821428572],
        [-2.4378694107142853, 1.2263075178571428, -1.1802564821428572],
        [-4.030917410714286, 1.620355517857143, 0.20542451785714283],
        [-3.3890224107142854, 2.086932517857143, 0.9581945178571428],
        [-4.786954410714286, 1.055215517857143, 0.7578315178571429],
        [-4.705796410714286, 2.697496517857143, -0.6323974821428572],
        [-5.372235410714286, 2.2562225178571427, -1.3759484821428571],
        [-5.2972994107142855, 3.3740205178571427, -0.014965482142857173],
        [-3.965870410714285, 3.295630517857143, -1.1684284821428572],
    ],
    "charge": 0,
    "spin": 1,
    "size": 0.0
}

global oxybenzone_data
oxybenzone_data = {
    "num_atoms": 29,
    "name": "oxybenzone",
    "atoms": ['C', 'C', 'C', 'C', 'C', 'C', 'H', 'C', 'O', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'H', 'O', 'C', 'H', 'H', 'H', 'O', 'H', 'H'],
    "coords": [
        [2.8035907241379308, -0.4830889310344828, 0.31171544827586206],
        [1.7467487241379311, -1.1601059310344828, 0.919205448275862],
        [0.4922377241379311, -0.5957459310344828, 0.9020494482758621],
        [0.23824372413793113, 0.6453840689655173, 0.3159804482758621],
        [1.3068487241379312, 1.3102510689655171, -0.2873885517241379],
        [2.5812027241379307, 0.7448800689655172, -0.2971815517241379],
        [3.385087724137931, 1.2793680689655171, -0.7846865517241379],
        [-1.1168162758620688, 1.2651360689655171, 0.4203244482758621],
        [-1.2606222758620689, 2.4213810689655175, 0.741730448275862],
        [-2.313361275862069, 0.40184506896551725, 0.1452914482758621],
        [-2.251433275862069, -0.7107779310344828, -0.6882195517241378],
        [-3.534515275862069, 0.7684500689655173, 0.705222448275862],
        [-3.394730275862069, -1.4473629310344829, -0.9568835517241381],
        [-1.3101642758620689, -0.9932989310344829, -1.140039551724138],
        [-4.671713275862069, 0.02369006896551723, 0.4503834482758621],
        [-3.5719842758620692, 1.6455220689655172, 1.336883448275862],
        [-4.603332275862068, -1.0852739310344828, -0.3828185517241379],
        [-3.341309275862069, -2.3041529310344826, -1.615228551724138],
        [-5.615068275862068, 0.30802206896551726, 0.8972854482758621],
        [-5.494159275862069, -1.6648629310344827, -0.5867205517241378],
        [1.933219724137931, -2.1131469310344824, 1.391999448275862],
        [4.004751724137932, -1.0934609310344827, 0.3572304482758621],
        [5.113187724137932, -0.45214493103448283, -0.22870755172413793],
        [5.963530724137931, -1.1085179310344828, -0.06605955172413791],
        [4.9706357241379315, -0.3100049310344828, -1.303546551724138],
        [5.310409724137932, 0.5144450689655172, 0.24278844827586207],
        [1.0733537241379312, 2.4912180689655172, -0.8947645517241378],
        [1.882667724137931, 2.8245570689655173, -1.2809085517241379],
        [-0.32650727586206885, -1.1222039310344827, 1.3750634482758621],
    ]
    ,
    "charge": 0,
    "spin": 1,
    "size": 0.0
}

global avobenzone_data
avobenzone_data = {
    "num_atoms": 45,
    "name": "avobenzone",
    "atoms": ['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'C', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'C', 'H', 'H', 'C', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'O', 'C', 'H', 'H', 'H', 'O', 'O'],
    "coords": [
        [-3.2471907111111107, -0.029338311111111015, 0.1320089111111111],
        [-2.515536711111111, 0.33976368888888897, -0.990419088888889],
        [-1.3286977111111107, 1.049899688888889, -0.8800080888888889],
        [-0.8368777111111108, 1.410781688888889, 0.3680959111111111],
        [-1.5619977111111107, 1.043048688888889, 1.5022439111111112],
        [-2.739998711111111, 0.34011868888888896, 1.3824579111111113],
        [-2.8607047111111106, 0.07535068888888899, -1.9788270888888888],
        [-0.7871387111111108, 1.3001076888888892, -1.7816810888888888],
        [-1.1803057111111106, 1.3209526888888892, 2.475009911111111],
        [-3.2772937111111107, 0.07149568888888896, 2.282448911111111],
        [-4.557832711111111, -0.8110383111111111, 0.0471959111111111],
        [-5.68265771111111, 0.016693688888888975, 0.6883079111111111],
        [-6.626711711111111, -0.528413311111111, 0.6334819111111111],
        [-5.481231711111111, 0.22989268888888897, 1.737937911111111],
        [-5.807573711111111, 0.9683996888888889, 0.1699959111111111],
        [-4.403201711111111, -2.139125311111111, 0.8047019111111111],
        [-5.331255711111111, -2.711176311111111, 0.751051911111111],
        [-3.6046927111111104, -2.7422903111111108, 0.3705299111111111],
        [-4.169711711111111, -1.980302311111111, 1.8573259111111111],
        [-4.956199711111111, -1.125447311111111, -1.3962720888888889],
        [-5.106825711111111, -0.21725031111111104, -1.9816000888888887],
        [-4.207291711111111, -1.739359311111111, -1.8985260888888889],
        [-5.894987711111111, -1.680617311111111, -1.4009420888888888],
        [0.42969228888888933, 2.170386688888889, 0.5539479111111111],
        [1.2380222888888892, 2.575069688888889, -0.677167088888889],
        [0.5865622888888893, 3.029724688888889, -1.4205450888888889],
        [1.9648132888888894, 3.3129756888888893, -0.3437100888888889],
        [1.9480212888888893, 1.412276688888889, -1.3710050888888887],
        [3.120996288888889, 0.7908566888888889, -0.7090350888888889],
        [3.6197622888888894, 1.219004688888889, 0.5248689111111111],
        [3.7539692888888894, -0.270231311111111, -1.349901088888889],
        [4.71393328888889, 0.604695688888889, 1.0917649111111112],
        [3.143627288888889, 2.027687688888889, 1.0616639111111112],
        [4.853690288888889, -0.894795311111111, -0.793587088888889],
        [3.3656892888888894, -0.601821311111111, -2.303181088888889],
        [5.340944288888889, -0.455538311111111, 0.4372619111111111],
        [5.102542288888889, 0.927746688888889, 2.047265911111111],
        [5.320558288888889, -1.714537311111111, -1.3184900888888889],
        [6.4033382888888895, -0.991211311111111, 1.0667899111111112],
        [7.080085288888889, -2.067315311111111, 0.4565679111111111],
        [7.8904582888888894, -2.333966311111111, 1.1291909111111111],
        [6.421497288888889, -2.929765311111111, 0.3262879111111111],
        [7.496049288888889, -1.778685311111111, -0.512051088888889],
        [0.8193432888888893, 2.478773688888889, 1.6568509111111112],
        [1.5523202888888894, 1.0265216888888888, -2.448307088888889],
    ]
    ,
    "charge": 0,
    "spin": 1,
    "size": 0.0

}

def define_fragments():
    frags = []

    frags.append(
        {
            "num_atoms": 19,
            "name": "arginine",
            "atoms": ['N', 'H', 'H', 'C', 'N', 'H', 'H', 'N', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'H'],
            "coords": [
                [-2.149366052631579, 1.280494894736842, 5.8421052631544205e-06],
                [-1.3771460526315789, 1.920837894736842, -0.00012415789473684558],
                [-3.074224052631579, 1.669389894736842, 0.00018684210526315442],
                [-1.9501300526315792, -0.03491610526315792, -2.5157894736845577e-05],
                [-3.000153052631579, -0.8577391052631579, -1.0157894736845579e-05],
                [-2.8825910526315788, -1.854114105263158, 0.00026584210526315444],
                [-3.940025052631579, -0.5069571052631578, 0.00010984210526315442],
                [-0.719979052631579, -0.519462105263158, -9.415789473684558e-05],
                [-0.605082052631579, -1.5188031052631579, -0.00027215789473684557],
                [0.501152947368421, 0.2875508947368421, -8.515789473684558e-05],
                [0.511472947368421, 0.9243478947368421, -0.8887071578947369],
                [0.5113809473684209, 0.924442894736842, 0.8884678421052632],
                [1.7269709473684212, -0.611826105263158, 2.684210526315442e-05],
                [1.6952549473684209, -1.2578831052631578, -0.8806161578947368],
                [1.695172947368421, -1.2577721052631579, 0.8807468421052632],
                [3.014574947368421, 0.20012289473684208, 2.9842105263154425e-05],
                [3.8796049473684207, -0.4595711052631579, 0.00011184210526315441],
                [3.081593947368421, 0.8358708947368421, -0.8833471578947368],
                [3.081516947368421, 0.835985894736842, 0.8833298421052632]
            ],
            "charge": 1,
            "spin": 1,
            "size": 0.0

        }
    )

    frags.append(
        {
            "num_atoms": 17,
            "name": "lysine",
            "atoms": ['N', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'H', 'H'],
            "coords": [
                [-0.1042588335294119, -2.00525728, 1.473917954705882],
                [0.22287441647058825, -2.95022911, 1.4704449947058822],
                [0.27274188647058795, -1.5249122800000001, 2.2658391247058822],
                [0.328762786470588, -1.3322867200000001, 0.2408312147058822],
                [-0.07462797352941175, -1.84625587, -0.6065244452941179],
                [1.397378736470588, -1.3395881200000002, 0.18691787470588217],
                [-0.17502241352941184, 0.1229699099999999, 0.2461795647058822],
                [-1.2436383635294117, 0.13027130999999992, 0.30009290470588224],
                [0.22836834647058835, 0.63693906, 1.093535224705882],
                [0.27861927647058815, 0.8279866899999999, -1.0456255952941178],
                [-0.12477148352941203, 0.31401754000000004, -1.8929812452941177],
                [1.347235226470588, 0.8206852799999997, -1.0995389252941177],
                [-0.22516592352941167, 2.28324332, -1.0402772352941176],
                [0.17822483647058807, 2.79721246, -0.19292158529411796],
                [0.09002667647058793, 2.77309263, -1.9378301752941178],
                [-1.293781873529412, 2.2905447199999998, -0.9863639052941178],
                [-1.102965323529412, -1.9984335400000002, 1.524304254705882]
            ],
            "charge":1,
            "spin": 1,
            "size": 0.0

        }
    )

    frags.append(
        {
            "num_atoms": 7,
            "name": "aspartic",
            "atoms": ['O', 'C', 'C', 'H', 'H', 'H', 'O'],
            "coords": [
                [1.6558742642857145, 0.7319832657142856, -0.34302586285714287],
                [0.4477709842857145, 0.7119481857142856, 0.008621177142857157],
                [-0.34072376571428564, -0.6105680342857143, -0.020134372857142844],
                [-0.24345327571428554, -1.1044111942857144, 0.9240887971428572],
                [-1.3735099757142855, -0.4059838542857144, -0.21091682285714283],
                [0.046941714285714475, -1.2402001942857144, -0.7935545728571428],
                [-0.19289994571428548, 1.9172318257142857, 0.43492165714285713]
            ],
            "charge": -1,
            "spin": 1,
            "size": 0.0
        }
    )

    frags.append(
        {
            "num_atoms": 7,
            "name": "glutamic",
            "atoms": ['C', 'O', 'O', 'C', 'H', 'H', 'H'],
            "coords": [
                [-0.40285092285714286, 0.7382373685714286, 0.012653958571428562],
                [0.19128581714285717, 1.8052306085714287, 0.3161297085714285],
                [-1.7932467028571428, 0.7639678785714287, -0.32056892142857146],
                [0.36740797714285717, -0.5952333614285714, 0.0001229585714285619],
                [0.3105925371428572, -1.0490497314285714, 0.9674505785714286],
                [-0.06517468285714284, -1.2501819914285714, -0.7270733914285714],
                [1.391985977142857, -0.4129707714285713, -0.24871489142857142]
            ],
            "charge": -1,
            "spin": 1,
            "size": 0.0
        }
    )

    frags.append(
        {
            "num_atoms": 14,
            "name": "isoleucine",
            "atoms": ['C', 'H', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'H'],
            "coords": [
                [-0.248106, 0.722549, 0.6713884285714286],
                [-1.342172, 0.724819, 0.6832444285714285],
                [0.248106, 1.548297, -0.5100675714285714],
                [0.248106, -0.722549, 0.6713884285714286],
                [1.339315, 1.545443, -0.5559105714285714],
                [-0.12406199999999999, 1.160183, -1.4588485714285715],
                [-0.07900799999999998, 2.585336, -0.4300755714285714],
                [1.342172, -0.724819, 0.6832444285714285],
                [-0.06799399999999999, -1.203132, 1.6002694285714285],
                [-0.248106, -1.548297, -0.5100675714285714],
                [0.12406200000000002, -1.160183, -1.4588485714285715],
                [0.07900800000000001, -2.585336, -0.4300755714285714],
                [-1.339315, -1.545443, -0.5559105714285714],
                [0.06799400000000001, 1.203132, 1.6002694285714285]
            ],
            "charge": 0,
            "spin": 1,
            "size": 0.0
        }
    )

    frags.append(
        {
            "num_atoms": 14,
            "name": "leucine",
            "atoms": ['C', 'H', 'H', 'C', 'H', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H'],
            "coords": [
                [-7.93016446160826e-18, 1.4499611428571428, -0.06375257142857141],
                [-7.93016446160826e-18, 1.4968431428571427, -1.1564665714285716],
                [0.883012, 1.9845861428571427, 0.2916454285714286],
                [-7.93016446160826e-18, 1.4285714282953016e-07, 0.4077454285714286],
                [-7.93016446160826e-18, 1.4285714282953016e-07, 1.5030394285714286],
                [-1.255703, -0.7249808571428572, -0.06375257142857141],
                [1.255703, -0.7249808571428572, -0.06375257142857141],
                [-1.296304, -0.7484218571428572, -1.1564665714285716],
                [-1.277196, -1.7570038571428572, 0.2916454285714286],
                [-2.160208, -0.22758185714285717, 0.2916454285714286],
                [2.160208, -0.22758185714285717, 0.2916454285714286],
                [1.277196, -1.7570038571428572, 0.2916454285714286],
                [1.296304, -0.7484218571428572, -1.1564665714285716],
                [-0.883012, 1.9845861428571427, 0.2916454285714286]
            ]
            ,
            "charge": 0,
            "spin": 1,
            "size": 0.0

        }
    )

    frags.append(
        {
            "num_atoms": 6,
            "name": "serine",
            "atoms": ['O', 'H', 'C', 'H', 'H', 'H'],
            "coords": [
                    [-0.1389911666666667, -1.0804205, 0.0],
                    [0.7654168333333333, -1.3877454999999999, 0.0],
                    [-0.1389911666666667, 0.33025350000000003, 0.0],
                    [-1.1797341666666665, 0.6498875, 0.0],
                    [0.3461498333333333, 0.7440125, 0.890351],
                    [0.3461498333333333, 0.7440125, -0.890351]
            ]
            ,
            "charge": 0,
            "spin": 1,
            "size": 0.0
        }
    )

    frags.append(
        {
            "num_atoms":19,
            "name": "tryptophan",
            "atoms": ['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'C', 'C', 'H', 'N', 'H', 'C', 'H', 'H', 'H'],
            "coords": [
                    [0.799817, -1.3618299473684212, -5.268421052631055e-05],
                    [-0.040227999999999986, -0.244916947368421, 1.3157894736894564e-06],
                    [0.532212, 1.042519052631579, -4.268421052631054e-05],
                    [1.910519, 1.2401040526315792, -8.868421052631054e-05],
                    [2.714339, 0.11930905263157897, 3.3315789473689454e-05],
                    [2.165274, -1.171611947368421, 6.431578947368946e-05],
                    [0.38329199999999997, -2.361199947368421, -1.7684210526310542e-05],
                    [2.338177, 2.234374052631579, -6.968421052631054e-05],
                    [3.789782, 0.23765805263157896, 0.00011731578947368945],
                    [2.827406, -2.027096947368421, 0.00015931578947368945],
                    [-1.4673239999999999, -0.08076594736842102, 8.431578947368946e-05],
                    [-1.692607, 1.2605200526315792, 6.931578947368945e-05],
                    [-2.629076, 1.7942460526315789, 0.00018831578947368945],
                    [-0.49810599999999994, 1.9450620526315792, 3.5315789473689455e-05],
                    [-0.40103599999999995, 2.940852052631579, -0.00019368421052631053],
                    [-2.482686, -1.173145947368421, -5.068421052631055e-05],
                    [-2.377944, -1.8117019473684208, -0.8797486842105263],
                    [-2.376954, -1.812779947368421, 0.8787403157894736],
                    [-3.494857, -0.769594947368421, 0.0007713157894736895]
            ]
            ,
            "charge": 0,
            "spin": 1,
            "size": 0.0
        }
    )

    frags.append(
        {
            "num_atoms": 17,
            "name": "tyrosine",
            "atoms": ['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'H', 'O', 'H'],
            "coords": [
                    [-1.329380882352941, -0.3750093529411765, 0.08929000000000001],
                    [-0.517843882352941, -1.4744843529411764, 0.059448],
                    [0.8537301176470589, -1.2297273529411765, 0.020797000000000003],
                    [1.355825117647059, 0.06915464705882356, 0.014831000000000004],
                    [0.4629491176470589, 1.1375616470588235, 0.043315000000000006],
                    [-0.914064882352941, 0.9278686470588235, 0.081821],
                    [-0.9034408823529411, -2.4851793529411763, 0.062572],
                    [1.5394461176470589, -2.0683683529411767, -0.006701999999999993],
                    [0.8427661176470589, 2.1524366470588236, 0.03379600000000001],
                    [-1.609510882352941, 1.7563506470588235, 0.10380900000000001],
                    [2.840171117647059, 0.3148846470588235, 0.0055610000000000034],
                    [3.226114117647059, 0.3900866470588235, 1.024221],
                    [3.373317117647059, -0.49622435294117645, -0.489159],
                    [3.084615117647059, 1.2443076470588235, -0.5077160000000001],
                    [-3.505750882352941, -0.18745435294117646, 0.274864],
                    [-4.37366288235294, 0.17744164705882354, 0.07189300000000001],
                    [-4.425278882352941, 0.14635464705882356, -0.882641]
            ]
            ,
            "charge": 0,
            "spin": 1,
            "size": 0.0
        }
    )

    frags.append(
        {
            "num_atoms": 15,
            "name": "phenylalanine",
            "atoms": ['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'C', 'H', 'H', 'H'],
            "coords": [
                    [-1.041578616666666, -1.803521892666667, 0.0006557966666666665],
                    [0.3535813833333341, -1.803521892666667, 0.0006557966666666665],
                    [1.0511193833333343, -0.5957708926666669, 0.0006557966666666665],
                    [0.3534653833333343, 0.6127381073333331, -0.0005432033333333322],
                    [-1.0413596166666657, 0.6126601073333332, -0.0010222033333333325],
                    [-1.7389606166666662, -0.5955458926666668, -2.6203333333332107e-05],
                    [-1.591337616666666, -2.7558388926666666, 0.0011057966666666655],
                    [0.903089383333334, -2.7560348926666665, 0.0019707966666666667],
                    [2.150799383333334, -0.5956908926666669, 0.0012897966666666656],
                    [-1.5914816166666657, 1.564941107333333, -0.001975203333333335],
                    [-2.8385646166666656, -0.595362892666667, -0.00020620333333333518],
                    [1.1239694533333342, 1.946126077333333, -0.0006258333333333324],
                    [1.322942303333334, 2.243143987333333, 1.0078830466666666],
                    [2.048388123333334, 1.823846937333333, -0.5254095333333333],
                    [0.535927903333334, 2.6978318173333333, -0.4844084433333333]
            ],
            "charge": 0,
            "spin": 1,
            "size": 0.0
        }
    )

    return frags

def get_fragment_cutoff(frag_to_calc: dict, vdw_frac: float):
    '''
        Accepts a fragment dictionary and calculates the spatial extent, aka,
        the minnimum distance between the fragment and any atoms of the binding site.

        Args:
            frag_to_calc: fragment dictionary to calculate
            vdw_frac: a vdW distance (fraction of the spatial extent) to add to the cartesian extent
        Returns:
            cutoff: the distance value
    '''
    formula, test_list = show_fragment(frag_to_calc)
    atoms1 = Atoms(formula, test_list)

    distances = []
    for i in range(frag_to_calc["num_atoms"]):
        for j in range(i+1,frag_to_calc["num_atoms"],1):
            dis_vec = atoms1.positions[i] - atoms1.positions[j]
            dis = np.linalg.norm(dis_vec)
            distances.append(float(dis))

    maxval = max(distances)
    cutoff = maxval*(0.5 + vdw_frac)
    frag_to_calc["size"] = cutoff
    print(f"cutoff is {cutoff}")

    return cutoff


def show_fragment(frag_to_show: dict):
    '''
        accepts a fragment dictionary and displays atoms list and coordinates

        Args:
            frag_to_show: a fragment dictionary object
        Returns:
            None; prints fragment info
    '''
    test_list = []
    formula = ""
    for i,atom in enumerate(frag_to_show["atoms"]):
        geotup = (frag_to_show["coords"][i][0],frag_to_show["coords"][i][1],frag_to_show["coords"][i][2])
        test_list.append(geotup)
        formula += atom

    print(str(formula))
    #print(test_list)

    return formula,test_list

def add_box(val, vdw_distance):
    '''
        Adds an extra vdW distance to a dimension

        Args:
            val: the existing dimension
            vdw_distance: distance to add
        Returns:
            new val: val + vdw distance
    '''
    new_val = val + vdw_distance
    return new_val

def sub_box(val, vdw_distance):
    '''
        Adds an extra vdW distance to a dimension

        Args:
            val: the existing dimension
            vdw_distance: distance to add
        Returns:
            new val: val - vdw distance
    '''
    new_val = val - vdw_distance
    return new_val


def get_binding_site_dims(all_molecules: dict, vdw_distance: float):
    '''
        Uses the array of atom positions to find the maximum and minimum dimensions

        Args:
            all_molecules: array of atom coordinates
            vdw_distance: extra distance to add to binding site extent
        Returns:
            max_values: np.array of maximum values
            min_values: np.array of minimum values
    '''
    max_values = np.zeros((3))
    min_values = np.zeros((3))
    x_list = []
    y_list = []
    z_list = []
    for row in all_molecules["coords"]:
        x_list.append(row[0])
        y_list.append(row[1])
        z_list.append(row[2])
    max_values[0] = np.max(x_list)
    max_values[1] = np.max(y_list)
    max_values[2] = np.max(z_list)
    min_values[0] = np.min(x_list)
    min_values[1] = np.min(y_list)
    min_values[2] = np.min(z_list)

    max_values = [add_box(val, vdw_distance) for val in max_values]
    min_values = [sub_box(val, vdw_distance) for val in min_values]

    dims = ["x","y","z"]
    print(f"Maximum dimensions after augmentation are:")
    for dim,maxes,mins in zip(dims,max_values,min_values):
        print(f"{dim} - Max: {maxes:10.3f}, Min: {mins:10.3f}")

    volume = 1.0
    for big,small in zip(max_values,min_values):
        volume *= (big - small)
    print(f"Volume is {volume} A^3")

    return max_values, min_values

def get_frag_coordinates(new_origin: list, which_frag: dict):
    """
        Given an origin set of coordinates for a new fragment, this function
        reads in the standard fragment coordinates, rotates them randomly
        in each direction, and then translates them to the new origin.

        Args:
            new_origin: the location of the new fragment
            which_frag: a dictionary for the fragment in question.
    """
    how_many_atoms = which_frag['num_atoms']
    theta_x = random.uniform(0, 2 * np.pi)
    theta_y = random.uniform(0, 2 * np.pi)
    theta_z = random.uniform(0, 2 * np.pi)
    x_rotation_matrix = np.asarray(([1.0, 0.0, 0.0], [0.0, np.cos(theta_x), -np.sin(theta_x)], [0.0, np.sin(theta_x), np.cos(theta_x)])).reshape(3, 3)
    y_rotation_matrix = np.asarray(([np.cos(theta_y), 0.0, -np.sin(theta_y)], [0.0, 1.0, 0.0], [np.sin(theta_y), 0.0, np.cos(theta_y)])).reshape(3, 3)
    z_rotation_matrix = np.asarray(([np.cos(theta_z), -np.sin(theta_z), 0.0], [np.sin(theta_z), np.cos(theta_z), 0.0], [0.0, 0.0, 1.0])).reshape(3, 3)
    atoms_list = []
    for vec in which_frag['coords']:
        temp_vec = np.array(vec)
        rot_temp = np.matmul(x_rotation_matrix, temp_vec)
        rot_temp = np.matmul(y_rotation_matrix, rot_temp)
        rot_temp = np.matmul(z_rotation_matrix, rot_temp)
        temp_vec = rot_temp + new_origin
        temp_vec = list(temp_vec)
        atoms_list.append(temp_vec)
    return atoms_list

def calc_frag_energy(new_molecule: list, frag: dict, atom_symbols: list, calculator, bs_charge, bs_spin, ligand_charge, ligand_spin, opt_flag):
    """
        Calculates the energies of the fragments in the binding site

        Args:
            new_molecules: list of arrays of coordinates of the binding site and fragment
            frag: the dictionary for the fragment in question
            atom_symbols: list of atom symbols
            calculator: ASE calculator
            bs_charge: binding site charge
            bs_spin: binding site spin
            ligand_charge: ligand charge
            ligand_spin: ligand spin
        Returns:
            ies: the interaction energies between the binding site and the fragment
    """
    path = 'temp_files/'
    test_files = ['complex.xyz']
    all_symbols = atom_symbols + frag['atoms']
    f = open(path + test_files[0], 'w')
    f.write(f'{len(all_symbols)}\n')
    f.write('\n')
    for i in range(len(new_molecule)):
        row_string = f'{all_symbols[i]}'
        for coord in new_molecule[i]:
            row_string += f'    {coord}'
        if i != len(new_molecule):
            f.write(row_string + '\n')
        else:
            f.write(row_string)
    f.close()
    atoms_tot = ase.io.read(path + test_files[0], format='xyz')
    total_spin = bs_spin + ligand_spin - 1
    total_charge = bs_charge + ligand_charge
    atoms_tot.info.update({'spin': total_spin, 'charge': total_charge})

    atoms_tot.calc = calculator
    initial_energy = atoms_tot.get_potential_energy()
    # print(f"Initial energy: {0.0367493*initial_energy:.6f} ha")

    bs_length = len(new_molecule) - frag['num_atoms']
    c = FixAtoms(indices = list(range(bs_length)))
    atoms_tot.set_constraint(c)
    if opt_flag:
      opt = BFGS(atoms_tot)
      opt.run(fmax=0.05)

    os.remove(path + test_files[0])

    atoms_bs = atoms_tot[:bs_length]
    atoms_bs.info.update({'spin': bs_spin, 'charge': bs_charge})
    atoms_l = atoms_tot[bs_length:]
    atoms_l.info.update({'spin': ligand_spin, 'charge': ligand_charge})
    atoms_to_calc = [atoms_tot, atoms_bs, atoms_l]
    results = []
    for atoms in atoms_to_calc:
        atoms.calc = calculator
        results.append(atoms.get_potential_energy())
    ie = 23.06035 * (results[0] - results[1] - results[2])

    ## convert ataos_tot to np array
    opt_molecule = atoms_tot.get_positions()
    opt_molecule = np.array(opt_molecule)

    return ie, opt_molecule

def grow_fragments(all_molecules: list, frags: list, number_tries: int, calculator, max_values: list, min_values: list):
    """
        Accepts the binding site coordinates and fragment dictionaries and
        tries to add the fragments to the binding site.

        Args:
            all_molecules: list of binding site atom coordinates
            frags: list of fragment dictionaries
            number_tries: how many times at attempt fragment placement
            calculator: ASE calculator
            bs_object: dictionary for the binding site
            max_values: np.array of maximum values
            min_values: np.array of minimum values
        Returns:
            new_molecules: list of positions of successfully placed fragments
    """
    try:
        os.mkdir('temp_files')
    except:
        print('temp_files directory already exists')
    new_molecules = []
    ies = []
    centers = []
    sigmas = []
    for i in range(3):
        centers.append((max_values[i] + min_values[i]) / 2)
        sigmas.append((max_values[i] - min_values[i]) / 4)
    for mi, molecule in enumerate(all_molecules):
        for frag in frags:
            print(f"Fragment: {frag['name']}")
            new_sheet = []
            ie_sheet = []
            how_many_added = 0
            for _ in range(number_tries):
                add_mol = True
                new_mol_origin = np.empty(3)
                cutoff = frag['size']
                for i in range(3):
                    new_mol_origin[i] = np.random.normal(loc=centers[i], scale=0.5*max_values[i])
                # for i in range(3):
                #     distance = abs(new_mol_origin[i] - centers[i])
                #     if distance < max_values[i]:
                #         add_mol = False
                #         break
                # for row in molecule["coords"]:
                #     distance = calc_distance(new_mol_origin, row)
                #     if distance < cutoff:
                #         add_mol = False
                #         break
                if add_mol:
                    new_vec = get_frag_coordinates(new_mol_origin, frag)
                    new_vec = np.array(new_vec)
                    single_frag = np.append(molecule["coords"], new_vec, axis=0)

                    ie, single_frag = calc_frag_energy(single_frag, frag, molecule['atoms'], calculator, molecule['charge'], molecule['spin'], frag['charge'], frag['spin'], opt_flag = False)
                    if ie < 200.0:
                        ie, single_frag = calc_frag_energy(single_frag, frag, molecule['atoms'], calculator, molecule['charge'], molecule['spin'], frag['charge'], frag['spin'], opt_flag = True)
                        #keep if new ie < 0
                        how_many_added += 1
                        print(f"adding fragment: {frag['name']}")
                        new_sheet.append(single_frag)
                        ie_sheet.append(ie)
            print(f"Added {how_many_added} {frag['name']} fragments")
            new_molecules.append(new_sheet)
            ies.append(ie_sheet)
    return (new_molecules, ies)

def calc_distance(ref, atom):
    """
        Calculates the distance between two points

        Args:
            ref: a reference point
            atom: the new atom coordinates
        Returns:
            distance: the distance
    """
    distance = 0.0
    for i in range(3):
        distance += (ref[i] - atom[i]) ** 2
    distance = np.sqrt(distance)
    return distance

def save_xyz_files(new_molecules: list, frags: list, molecule_dict):
    '''
        accepts the new_molecules array and creates an XYZ file for each fragment placement

        Args:
            new_molecules: list of arrays of coordinates of the binding site and fragment
            frag: the dictionary for the fragment in question
            bs_object: dictionary for the binding site
            atom_symbols: list of atom symbols
        Returns:
            None; saves XYZ files
    '''
    try:
      os.mkdir("frag_files")
    except:
      print("frag_files directory already exists")

    try:
        files = os.listdir("frag_files")
        files_to_remove = [file for file in files if (os.path.splitext(file)[1]==".xyz")]
        for file in files_to_remove:
          os.remove(f"frag_files/{file}")
    except:
        print("frag_files directory is empty")

    for k,frag in enumerate(frags):
        for j in range(len(new_molecules[k])):
            mol_file = f"frag_files/{molecule_dict['name']}_w_{frag['name']}{j}.xyz"

            all_symbols = molecule_dict["atoms"] + frag["atoms"]
            f = open(mol_file,"w")
            f.write(f"{len(all_symbols)}\n")
            f.write("\n")
            for i in range(len(new_molecules[k][j])):
                row_string = f"{all_symbols[i]}"
                for coord in new_molecules[k][j][i]:
                    row_string += f"    {coord}"
                if i != len(new_molecules[k][j]):
                    f.write(row_string+"\n")
                else:
                    f.write(row_string)
            f.close()
            #print("File Written")

        print(f"{len(new_molecules[k])} files written for {frag['name']}.")

def view_frag_pose(frag_idx: int, pose_idx: int, frags: list, bs: dict):
    '''
      Displays a fragment pose

      Args:
        frag_idx: index of the fragment to view
        pose_idx: index of the pose to view
        frags: list of fragment dictionaries
        bs: dictionary for the binding site
      Returns:
        None; displays fragment pose
    '''
    frag_name = frags[frag_idx]["name"]
    view_file = mol_file = f"frag_files/{bs['name']}_w_{frag_name}{pose_idx}.xyz"
    f = open(view_file,"r")
    lines = f.readlines()
    mol_data = "".join(lines)
    f.close

    viewer = py3Dmol.view(width=800, height=400)
    viewer.addModel(mol_data, "xyz")  # Add the trajectory frame

    for i in range(len(bs["atoms"])):
      viewer.setStyle({'model': -1, 'serial': i}, {"stick": {}, "sphere": {"radius": 0.5}})

    for i in range(len(bs["atoms"]),len(bs['atoms'])+frags[frag_idx]["num_atoms"],1):
      viewer.setStyle({'model': -1, 'serial': i}, {"stick": {}, "sphere": {"radius": 0.5}}) #, 'color': 'green'}})

    viewer.zoomTo()
    viewer.show()

def get_best_poses(ies: list, frags: list):
    '''
      Accepts the list of interaction energies and determines the best pose for each fragment

      Args:
        ies: list of interaction energies
        frags: list of fragment dictionaries
      Returns:
        best_pose_for_fragments: list of best poses for each fragment
    '''
    best_pose_for_fragments = []
    for energy,frag in zip(ies,frags):
        try:
          min = np.min(energy)
          min_idx = np.argmin(energy)
          best_pose_for_fragments.append(min_idx)
          print(f"best pose for {frag['name']} is: {min:.3f} at location: {min_idx}")
        except:
          best_pose_for_fragments.append(-1)
          print(f"no poses for {frag['name']}")

    # for i,elist in enumerate(ies):
    #   if len(elist)==0:
    #     ies[i][0] = 0.0


    return best_pose_for_fragments

def get_best_seperate_poses(ies: list, frags: list, bs_object, new_molecules):
    '''
      Accepts the list of interaction energies and determines the best pose for each fragment

      Args:
        ies: list of interaction energies
        frags: list of fragment dictionaries
      Returns:
        best_pose_for_fragments: list of best poses for each fragment
    '''
    best_pose_for_fragments = []
    idx_frag = 0
    idx_used_frag = 0
    for energy,frag in zip(ies,frags):
        try:
          if idx_used_frag == 0:
            min = np.min(energy)
            min_idx = np.argmin(energy)
            best_pose_for_fragments.append(min_idx)
            print(f"best pose for {frag['name']} is: {min:.3f} at location: {min_idx}")
            idx_frag = idx_frag + 1
            idx_used_frag = idx_used_frag + 1
          else:
            min = np.min(energy)
            min_idx = np.argmin(energy)
            frag_coords = new_molecules[idx_frag][min_idx][bs_object["num_atoms"]:]
            previous_frag_coords = new_molecules[idx_frag-1][min_idx][bs_object["num_atoms"]:]
            distance_too_short_flag = distance_too_short(previous_frag_coords, frag_coords)
            while distance_too_short_flag == True:
              _ = energy.pop(min_idx)
              min = np.min(energy)
              min_idx = np.argmin(energy)
              frag_coords = new_molecules[idx_frag][min_idx][bs_object["num_atoms"]:]
              previous_frag_coords = new_molecules[idx_frag-1][min_idx][bs_object["num_atoms"]:]
              distance_too_short_flag = distance_too_short(previous_frag_coords, frag_coords)
            best_pose_for_fragments.append(min_idx)
            print(f"best pose for {frag['name']} is: {min:.3f} at location: {min_idx}")
            idx_frag = idx_frag + 1
            idx_used_frag = idx_used_frag + 1
        except:
          best_pose_for_fragments.append(-1)
          idx_frag = idx_frag + 1
          print(f"no poses for {frag['name']}")

    return best_pose_for_fragments


def distance_too_short(self, binding_site, ligand):
    '''
      Calculate the distance between atoms and return True if they are closer than the
      cutoff value of 1.4

        Args:
          bindingsite: list of binding site atom corrdinates
          ligand: list of ligand atom coordinates
        Returns:
          True if the distance is less than 1.4, False otherwise
    '''
    for row in binding_site:
      for conf_row in ligand:
        atom_atom_distance = self.calc_distance(row,conf_row)
        if atom_atom_distance < 1.4:
          return True
    return False

def combine_best_poses(frags: list, bs: dict, new_molecules: list, best_pose_for_fragments: list):
  '''
    Accepts the new_molecules array and combines the best poses for each fragment

    Args:
      frags: list of fragment dictionaries
      bs: dictionary for the binding site
      new_molecules: list of arrays of coordinates of the binding site and fragment
      best_pose_for_fragments: list of best poses for each fragment
    Returns:
      combined_poses: list of combined best poses
      combined_atoms: list of combined atoms
      (also generates the combined.xyz file in the frag_files/ folder)
  '''
  combined_poses = []
  combined_atoms = []

  centre_molecule_flag = True

  all_centers = []
  all_names = []
  for idx_frag, frag in enumerate(frags):
      try:
        if best_pose_for_fragments[idx_frag] == -1:
          continue
        if centre_molecule_flag:
          total_molecule = new_molecules[idx_frag][best_pose_for_fragments[idx_frag]]
          total_molecule = list(total_molecule)
          just_frag_array = total_molecule[bs["num_atoms"]:]
          xc, yc, zc = 0, 0, 0
          for row in just_frag_array:
            xc += row[0]
            yc += row[1]
            zc += row[2]
          xc /= len(just_frag_array)
          yc /= len(just_frag_array)
          zc /= len(just_frag_array)
          center = [xc, yc, zc]
          all_centers.append(center)
          all_names.append(frag["name"])
          combined_poses = [*total_molecule]
          combined_atoms = [*bs["atoms"], *frag["atoms"]]
          centre_molecule_flag = False
        else:
          total_molecule = new_molecules[idx_frag][best_pose_for_fragments[idx_frag]]
          just_frag_array = total_molecule[bs["num_atoms"]:].tolist()
          xc, yc, zc = 0, 0, 0
          for row in just_frag_array:
            xc += row[0]
            yc += row[1]
            zc += row[2]
          xc /= len(just_frag_array)
          yc /= len(just_frag_array)
          zc /= len(just_frag_array)
          center = [xc, yc, zc]
          all_centers.append(center)
          all_names.append(frag["name"])
          just_frag_array = list(just_frag_array)
          combined_poses = [*combined_poses, *just_frag_array]
          combined_atoms = [*combined_atoms, *frag["atoms"]]
      except:
        print(f"no best pose for {frag['name']}")



  path = "frag_files/"
  test_files = ["combined.xyz"]

  all_symbols = combined_atoms

  f = open(path+test_files[0],"w")
  f.write(f"{len(all_symbols)}\n")
  f.write("\n")

  for i in range(len(combined_poses)):
      row_string = f"{all_symbols[i]}"
      for coord in combined_poses[i]:
          row_string += f"    {coord}"
      if i != len(combined_poses):
          f.write(row_string+"\n")
      else:
          f.write(row_string)
  f.close()

  return combined_poses, combined_atoms, all_centers, all_names

def view_combined_poses(bs: dict, frags: list, best_poses: np.ndarray):
  '''
    Displays the combined best poses

    Args:
      bs: dictionary for the binding site
      frags: list of fragment dictionaries
    Returns:
      None; displays fragment pose
  '''

  view_file = "frag_files/combined.xyz"
  lines = []

  f = open(view_file,"r")
  lines1 = f.readlines()

  # lines.append(f"{int(lines1[0]) + bs['num_atoms']}\n")
  # lines.append("\n")

  # for atom, line in zip(bs["atoms"], bs["coords"]):
  #   line = f"{atom}"
  #   for coord in line:
  #     line += f"    {coord}"
  #   line += "\n"
  #   lines.append(line)
  # for line in lines1[2:]:
  #   lines.append(line)

  f.close
  mol_data = "".join(lines1)

  total_frags_length = 0
  for frag in frags:
    if best_poses[frags.index(frag)] == -1:
      continue
    total_frags_length += frag["num_atoms"]

  viewer = py3Dmol.view(width=800, height=400)
  viewer.addModel(mol_data, "xyz")  # Add the trajectory frame

  for i in range(bs["num_atoms"]):
    viewer.setStyle({'model': -1, 'serial': i}, {"stick": {}, "sphere": {"radius": 0.5}})

  for i in range(bs["num_atoms"],bs['num_atoms']+total_frags_length,1):
    viewer.setStyle({'model': -1, 'serial': i}, {"stick": {}, "sphere": {"radius": 0.5}, "color": 'green'})
  viewer.zoomTo()
  viewer.show()

def distance_between_centers(all_centers:list, all_names:list):
  distances = []
  for i in range(1, len(all_centers), 1):
# calculates distance between i and i-1 columns
    dist = calc_distance(all_centers[i], all_centers[i-1])
    distances.append(dist)
  sequence = all_names[0] + "-"
  for i, distance in enumerate(distances):
    print(f"distance between {all_names[i]}, {all_names[i+1]} is {distance:.2f}")
    increment = round(distance/3.8)
    if increment > 1:
      print(f"add {increment-1} linkers")
      for j in range (increment-1):
        sequence += "glycine - "
    sequence += f"{all_names[i+1]} - "
  print(sequence)
  return distances